# FinS4 BuildCorpus — S4 Chunks + True Training Pairs

## What this notebook does
1. Extract text from all 41 financial PDFs across 4 categories
2. Build S4 token-exact chunks (200 tokens / 50 overlap) → `s4_corpus.csv`
3. Generate 1 question per chunk using LLaMA via OpenRouter → `training_pairs.csv`

## Why chunk → question (industry standard)
Each question is generated **from** the chunk, so the (question, chunk) pair is a guaranteed true positive.
Previous approach mapped existing questions to nearest chunk via cosine similarity — noisy and unreliable.

## Resume support
If the cell is interrupted, re-run it — it will skip already-processed chunks and continue from where it stopped.

In [17]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────────
import sys, os
from pathlib import Path

_root = Path().resolve()
for _ in range(6):
    if (_root / 'config.py').exists():
        break
    _root = _root.parent
os.chdir(_root)
sys.path.insert(0, str(_root))

from config import (
    CONSUMER_PROTECTION_PATH,
    PAYMENT_INDUSTRY_PATH,
    REGULATORY_PATH,
    COMPLAINT_PROCEDURES_PATH,
)

FINE_TUNE_DIR  = 'fine_tuning'
S4_CORPUS_PATH = os.path.join(FINE_TUNE_DIR, 's4_corpus.csv')
PAIRS_PATH     = os.path.join(FINE_TUNE_DIR, 'training_pairs.csv')
MINILM_NAME    = 'sentence-transformers/all-MiniLM-L6-v2'

CATEGORY_DIRS = {
    'Regulatory'          : REGULATORY_PATH,
    'Consumer_Protection' : CONSUMER_PROTECTION_PATH,
    'Payment_Industry'    : PAYMENT_INDUSTRY_PATH,
    'Synthetic_Policies'  : COMPLAINT_PROCEDURES_PATH,
}

os.makedirs(FINE_TUNE_DIR, exist_ok=True)
os.makedirs(os.path.join(FINE_TUNE_DIR, 'Results'), exist_ok=True)

print('Repo root:', _root)
for cat, path in CATEGORY_DIRS.items():
    pdfs = list(Path(path).glob('*.pdf'))
    print(f'  [{cat:<22}]  {len(pdfs)} PDFs')

Repo root: /Users/momo/Desktop/GEN AI/Finsearch Project/FinSearch_Intent_Aware_Financial_Document_Intelligence-
  [Regulatory            ]  5 PDFs
  [Consumer_Protection   ]  5 PDFs
  [Payment_Industry      ]  5 PDFs
  [Synthetic_Policies    ]  26 PDFs


In [18]:
# ── Cell 2: Install ───────────────────────────────────────────────────────────
import sys
!{sys.executable} -m pip install -q pdfplumber transformers sentence-transformers faiss-cpu pandas tqdm openai python-dotenv

In [19]:
# ── Cell 3: Imports ───────────────────────────────────────────────────────────
import re
import time
import numpy as np
import pandas as pd
import pdfplumber
from tqdm.auto import tqdm
from pathlib import Path
from transformers import AutoTokenizer
from openai import OpenAI
from dotenv import load_dotenv

print('Imports OK')

Imports OK


In [20]:
# ── Cell 4: Build S4 Corpus from all PDFs ─────────────────────────────────────
# Skip if already built
if os.path.exists(S4_CORPUS_PATH):
    corpus_df = pd.read_csv(S4_CORPUS_PATH)
    print(f'S4 corpus already exists — loaded {len(corpus_df):,} chunks')
    print(corpus_df['category'].value_counts().to_string())
else:
    tokenizer = AutoTokenizer.from_pretrained(MINILM_NAME)

    def extract_pdf_text(pdf_path):
        parts = []
        try:
            with pdfplumber.open(pdf_path) as pdf:
                for page in pdf.pages:
                    raw = page.extract_text() or ''
                    raw = re.sub(r'\s+', ' ', raw).strip()
                    raw = re.sub(r'[^\x00-\x7F]+', ' ', raw)
                    if len(raw) > 50:
                        parts.append(raw)
        except Exception as e:
            print(f'  Warning: {Path(pdf_path).name}: {e}')
        return ' '.join(parts)

    def chunk_token_exact(text, tokenizer, target=200, overlap=50):
        ids  = tokenizer.encode(text, add_special_tokens=False)
        step = target - overlap
        chunks = []
        for start in range(0, len(ids), step):
            chunk_ids = ids[start:start + target]
            if len(chunk_ids) < 20:
                break
            txt = tokenizer.decode(chunk_ids, skip_special_tokens=True).strip()
            chunks.append((txt, len(chunk_ids)))
        return chunks

    records = []
    for cat, cat_dir in CATEGORY_DIRS.items():
        pdfs = sorted(Path(cat_dir).glob('*.pdf'))
        print(f'[{cat}] {len(pdfs)} PDFs')
        for pdf_path in tqdm(pdfs, desc=cat):
            text = extract_pdf_text(str(pdf_path))
            if not text:
                continue
            stem   = re.sub(r'[^a-z0-9]', '_', pdf_path.stem.lower())[:20]
            chunks = chunk_token_exact(text, tokenizer)
            for i, (chunk_text, token_count) in enumerate(chunks):
                records.append({
                    'chunk_id'   : f's4_{cat[:3].lower()}_{stem}_{i:04d}',
                    'category'   : cat,
                    'source_file': pdf_path.name,
                    'text'       : chunk_text,
                    'token_count': token_count,
                })

    corpus_df = pd.DataFrame(records)
    corpus_df.to_csv(S4_CORPUS_PATH, index=False)
    print(f'\nS4 corpus built: {len(corpus_df):,} chunks')
    print(corpus_df['category'].value_counts().to_string())
    print('Saved:', S4_CORPUS_PATH)

S4 corpus already exists — loaded 14,502 chunks
category
Regulatory             8027
Payment_Industry       5660
Consumer_Protection     632
Synthetic_Policies      183


In [21]:
# ── Cell 5: Generate Questions FROM Chunks (Resume-safe) ──────────────────────
#
# Uses YOUR OpenRouter key from .env (override=True ignores any shell env vars)
# Resume: reads existing training_pairs.csv, skips already-processed chunk_ids
# Checkpoint: saves every 50 chunks so progress is never lost

# ── Load your key ─────────────────────────────────────────────────────────────
load_dotenv(os.path.join(str(_root), '.env'), override=True)
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')
if not OPENROUTER_API_KEY:
    raise EnvironmentError('OPENROUTER_API_KEY not found in .env')
print(f'Key loaded — ending in: ...{OPENROUTER_API_KEY[-8:]}')  # verify it is YOUR key

client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url='https://openrouter.ai/api/v1',
)
MODEL = 'meta-llama/llama-3.3-70b-instruct'

# ── Load corpus ───────────────────────────────────────────────────────────────
corpus_df = pd.read_csv(S4_CORPUS_PATH)
print(f'Full corpus: {len(corpus_df):,} chunks')

# ── Resume: skip already-processed chunk_ids ──────────────────────────────────
if os.path.exists(PAIRS_PATH):
    existing_df  = pd.read_csv(PAIRS_PATH)
    done_ids     = set(existing_df['chunk_id'].tolist())
    pair_records = existing_df.to_dict('records')
    print(f'Resuming   : {len(done_ids):,} chunks already done, {len(pair_records):,} pairs saved')
else:
    done_ids     = set()
    pair_records = []
    print('Starting fresh')

remaining_df = corpus_df[~corpus_df['chunk_id'].isin(done_ids)].reset_index(drop=True)
print(f'Remaining  : {len(remaining_df):,} chunks to process')
print(f'Est. time  : ~{len(remaining_df) * 0.3 / 60:.0f} min')

# ── Question generation ───────────────────────────────────────────────────────
def generate_question(chunk_text, client, model):
    prompt = (
        'You are a financial document expert.\n'
        'Generate exactly 1 specific question that can be answered '
        'ONLY using the passage below. Be specific, use financial terminology.\n\n'
        f'Passage:\n{chunk_text[:800]}\n\n'
        'Output ONLY the question, no numbering, no explanation.'
    )
    try:
        r = client.chat.completions.create(
            model=model,
            messages=[{'role': 'user', 'content': prompt}],
            max_tokens=80,
            temperature=0.3,
        )
        q = r.choices[0].message.content.strip()
        q = re.sub(r'^\d+[\.)\s]\s*', '', q).strip()
        return q if len(q.split()) >= 5 else None
    except Exception as e:
        print(f'  API error: {e}')
        return None

# ── Main loop ─────────────────────────────────────────────────────────────────
CHECKPOINT = 50
rows       = remaining_df.to_dict('records')

for i, row in enumerate(tqdm(rows, desc='Generating')):
    q = generate_question(row['text'], client, MODEL)
    if q:
        pair_records.append({
            'question'   : q,
            'chunk_text' : row['text'],
            'chunk_id'   : row['chunk_id'],
            'category'   : row['category'],
            'source_file': row['source_file'],
        })
    time.sleep(0.3)
    if (i + 1) % CHECKPOINT == 0:
        pd.DataFrame(pair_records).to_csv(PAIRS_PATH, index=False)
        print(f'  [{i+1}/{len(rows)}] {len(pair_records):,} total pairs saved')

pairs_df = pd.DataFrame(pair_records)
pairs_df.to_csv(PAIRS_PATH, index=False)

print(f'\n=== Done ===')
print(f'Total pairs : {len(pairs_df):,}')
print(f'Saved to    : {PAIRS_PATH}')
print('\nPer-category:')
print(pairs_df['category'].value_counts().to_string())
print('\nSample:')
s = pairs_df.sample(1).iloc[0]
print(f'  Q    : {s["question"]}')
print(f'  Chunk: {s["chunk_text"][:150]}...')

Key loaded — ending in: ...0cc27214
Full corpus: 14,502 chunks
Resuming   : 3,531 chunks already done, 3,531 pairs saved
Remaining  : 10,971 chunks to process
Est. time  : ~55 min


Generating:   0%|          | 50/10971 [01:25<4:52:08,  1.61s/it]

  [50/10971] 3,581 total pairs saved


Generating:   1%|          | 100/10971 [03:11<4:52:17,  1.61s/it]

  [100/10971] 3,631 total pairs saved


...

  [10900/10971] 14,392 total pairs saved


Generating: 100%|█████████▉| 10950/10971 [7:15:41<00:39,  1.87s/it]

  [10950/10971] 14,442 total pairs saved


Generating: 100%|██████████| 10971/10971 [7:16:19<00:00,  2.39s/it]


=== Done ===
Total pairs : 14,463
Saved to    : fine_tuning/training_pairs.csv

Per-category:
category
Regulatory             8007
Payment_Industry       5645
Consumer_Protection     629
Synthetic_Policies      182

Sample:
  Q    : What method can entities consider to document roles and responsibilities as part of communicating day-to-day responsibilities for performing all the responsibilities allocated in requirement 11?
  Chunk: not occur. documented and assigned. good practice 11. 1. 2. b interview personnel with responsibility for roles and responsibilities may be documented...


In [22]:
# ── Cell 6: Quality Filter ────────────────────────────────────────────────────
pairs_df = pd.read_csv(PAIRS_PATH)
n_before = len(pairs_df)
print(f'Before filter: {n_before:,}')

pairs_df = pairs_df[pairs_df['question'].apply(lambda q: len(str(q).split()) >= 6)]
pairs_df = pairs_df.drop_duplicates(subset=['question'])
pairs_df = pairs_df.dropna(subset=['question', 'chunk_text', 'chunk_id'])

print(f'After filter : {len(pairs_df):,}  (removed {n_before - len(pairs_df):,})')
print('\nFinal category distribution:')
print(pairs_df['category'].value_counts().to_string())

pairs_df.to_csv(PAIRS_PATH, index=False)
print(f'\nSaved: {PAIRS_PATH}')

Before filter: 14,463
After filter : 13,996  (removed 467)

Final category distribution:
category
Regulatory             7941
Payment_Industry       5244
Consumer_Protection     629
Synthetic_Policies      182

Saved: fine_tuning/training_pairs.csv


In [ ]:
# ── Cell 7: Augment Minority Classes ─────────────────────────────────────────
#
# Target: ~2,000 pairs per category
# Regulatory (8007) & Payment_Industry (5645) → already enough, skip
# Consumer_Protection (629)  → generate 3 more q/chunk → ~2,500 total
# Synthetic_Policies  (182)  → generate 10 more q/chunk → ~2,000 total
#
# Each call generates DIFFERENT questions from the same chunk
# (temperature=0.7 ensures variety) — genuine new signal, not duplicates

# ── Reload key + client ───────────────────────────────────────────────────────
load_dotenv(os.path.join(str(_root), '.env'), override=True)
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')
if not OPENROUTER_API_KEY:
    raise EnvironmentError('OPENROUTER_API_KEY not found in .env')
print(f'Key loaded — ending in: ...{OPENROUTER_API_KEY[-8:]}')

client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url='https://openrouter.ai/api/v1',
)
MODEL = 'meta-llama/llama-3.3-70b-instruct'

# ── Config ────────────────────────────────────────────────────────────────────
AUGMENT_CONFIG = {
    'Consumer_Protection' : 3,   # 629 chunks × 3 extra = ~1,887 new → total ~2,500
    'Synthetic_Policies'  : 10,  # 182 chunks × 10 extra = ~1,820 new → total ~2,000
}

corpus_df = pd.read_csv(S4_CORPUS_PATH)
pairs_df  = pd.read_csv(PAIRS_PATH)

print('Current distribution:')
print(pairs_df['category'].value_counts().to_string())
print(f'\nWill augment: {list(AUGMENT_CONFIG.keys())}')

# ── Multi-question generation (higher temperature for variety) ────────────────
def generate_multiple_questions(chunk_text, client, model, n):
    prompt = (
        f'You are a financial document expert.\n'
        f'Generate exactly {n} DIFFERENT specific questions that can each be answered '
        f'ONLY using the passage below.\n'
        f'Requirements:\n'
        f'- Every question must be answerable from this passage alone\n'
        f'- Questions must cover DIFFERENT aspects of the passage\n'
        f'- Be specific, use financial terminology\n'
        f'- No generic or repetitive questions\n\n'
        f'Passage:\n{chunk_text[:800]}\n\n'
        f'Output ONLY a numbered list:\n'
        + '\n'.join([f'{i}. [question]' for i in range(1, n+1)])
    )
    try:
        r = client.chat.completions.create(
            model=model,
            messages=[{'role': 'user', 'content': prompt}],
            max_tokens=50 * n,
            temperature=0.7,   # higher temp = more diverse questions
        )
        raw = r.choices[0].message.content.strip()
        questions = []
        for line in raw.split('\n'):
            line = line.strip()
            m = re.match(r'^\d+[\.)\s]\s*(.+)', line)
            if m:
                q = m.group(1).strip()
                if len(q.split()) >= 5:
                    questions.append(q)
        return questions[:n]
    except Exception as e:
        print(f'  API error: {e}')
        return []

# ── Augment ───────────────────────────────────────────────────────────────────
new_records = []
CHECKPOINT  = 30

for category, n_extra in AUGMENT_CONFIG.items():
    cat_chunks = corpus_df[corpus_df['category'] == category].reset_index(drop=True)
    print(f'\n[{category}] {len(cat_chunks)} chunks × {n_extra} extra questions')
    print(f'  Est. time: ~{len(cat_chunks) * 0.4 / 60:.0f} min')

    for i, row in enumerate(tqdm(cat_chunks.to_dict('records'), desc=category)):
        questions = generate_multiple_questions(row['text'], client, MODEL, n=n_extra)
        for q in questions:
            new_records.append({
                'question'   : q,
                'chunk_text' : row['text'],
                'chunk_id'   : row['chunk_id'],
                'category'   : row['category'],
                'source_file': row['source_file'],
            })
        time.sleep(0.4)
        if (i + 1) % CHECKPOINT == 0:
            # Append new records to existing and save
            augmented_df = pd.concat(
                [pairs_df, pd.DataFrame(new_records)], ignore_index=True
            )
            augmented_df.to_csv(PAIRS_PATH, index=False)
            print(f'  Checkpoint [{i+1}/{len(cat_chunks)}] — {len(new_records)} new pairs so far')

# ── Merge + deduplicate + save ────────────────────────────────────────────────
augmented_df = pd.concat([pairs_df, pd.DataFrame(new_records)], ignore_index=True)
augmented_df = augmented_df.drop_duplicates(subset=['question']).reset_index(drop=True)
augmented_df.to_csv(PAIRS_PATH, index=False)

print(f'\n=== Augmentation Done ===')
print(f'Pairs before : {len(pairs_df):,}')
print(f'New pairs    : {len(new_records):,}')
print(f'Total after  : {len(augmented_df):,}')
print('\nFinal distribution:')
print(augmented_df['category'].value_counts().to_string())